# 🎫 Customer Support Ticket Resolution Pipeline
**Environment:** Azure Databricks · PySpark · ADLS Gen2  
**Pipeline Design:** Medallion Architecture (Bronze → Silver → Gold)

## ⚙️ Step 1 · Spark Config & Connection to Azure Data Lake (ADLS Gen2)
In this step, we establish a connection between Azure Databricks and our Azure Storage Account `stcustomersupportkeshav` using the Storage Account Access Key.

In [0]:
STORAGE_ACCOUNT_NAME = "stcustomersupportkeshav"
STORAGE_ACCOUNT_KEY = "0nc9Sq1TP0XLPOetK7All5XGUvPf/eXScvhrxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

print("✅ Azure Storage credentials configured successfully.")

✅ Azure Storage credentials configured successfully.


## 🥉 Step 2 · Bronze Ingestion (Raw CSV to Spark DataFrames)
We read the Bronze CSV files (`Day1_Bronze.csv`, `Day2_Bronze.csv`, and `Agents_Bronze.csv`) which were copied from the `raw` container to the `bronze` container by Azure Data Factory. 
We also add a literal column `Day` to distinguish between Day 1 and Day 2 tickets.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Reading Day 1 tickets
bronze_day1 = (spark.read
               .option("header", "true")
               .option("inferSchema", "true")
               .csv(f"abfss://bronze@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/Day1_Bronze.csv")
               .withColumn("Day", F.lit(1).cast(IntegerType())))

# Reading Day 2 tickets
bronze_day2 = (spark.read
               .option("header", "true")
               .option("inferSchema", "true")
               .csv(f"abfss://bronze@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/Day2_Bronze.csv")
               .withColumn("Day", F.lit(2).cast(IntegerType())))

# Reading Agent Profiles mapping table
bronze_profiles = (spark.read
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .csv(f"abfss://bronze@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/Agents_Bronze.csv"))

# Printing counts to confirm ingestion
print(f"🥉 Bronze Ingestion Summary:")
print(f"   Day 1 tickets loaded : {bronze_day1.count()} rows")
print(f"   Day 2 tickets loaded : {bronze_day2.count()} rows")
print(f"   Agent profiles loaded: {bronze_profiles.count()} rows")

🥉 Bronze Ingestion Summary:
   Day 1 tickets loaded : 1006 rows
   Day 2 tickets loaded : 806 rows
   Agent profiles loaded: 50 rows


## 🥈 Step 3 · Silver Layer (Cleaning, Processing & Filtering)
In this section, we clean and validate our raw data. We apply the following rules:
1. **Rule R-5 (Critical Null Check)**: Drop rows missing `ticket_id`, `agent_id`, or `resolution_time`.
2. **Rule R-1 & R-2 (Resolution Time conversion and rounding)**: Convert text time like `"0h 22m 45s"` to integer minutes. Round up if seconds $\ge$ 30, else discard seconds.
3. **Rule R-4 (Scope Filter)**: Only include agents reporting to Team Leads TL01 through TL08.
4. **Rule R-3 (Quality Threshold)**: Keep only resolved tickets that took strictly more than 15 minutes.

### 3-A · Data Quality Gate (Rule R-5)
Drop any records that have null or empty critical keys.

In [0]:
def drop_critical_nulls(df, label: str):
    before = df.count()
    clean = df.filter(
        F.col("ticket_id").isNotNull()       & (F.col("ticket_id")       != "") &
        F.col("agent_id").isNotNull()        & (F.col("agent_id")        != "") &
        F.col("resolution_time").isNotNull() & (F.col("resolution_time") != "")
    )
    after = clean.count()
    dropped = before - after
    print(f"   [R-5 Null Filter] {label} | Row count: {before} -> {after} (Dropped {dropped} invalid rows)")
    return clean

print("🔍 Applying Rule R-5 (Data Quality Gate):")
clean_day1 = drop_critical_nulls(bronze_day1, "Day 1")
clean_day2 = drop_critical_nulls(bronze_day2, "Day 2")

🔍 Applying Rule R-5 (Data Quality Gate):
   [R-5 Null Filter] Day 1 | Row count: 1006 -> 1006 (Dropped 0 invalid rows)
   [R-5 Null Filter] Day 2 | Row count: 806 -> 806 (Dropped 0 invalid rows)


### 3-B · UDF for Parsing & Rounding Resolution Time (Rules R-1 & R-2)
Convert `"Xh Xm Xs"` text formats to total minutes with rounding.

In [0]:
import re

def parse_resolution_time(time_str: str):
    """
    Converts 'Xh Xm Xs' format into total minutes with rounding.
    If seconds >= 30, it rounds up.
    Returns None for invalid or unparseable text.
    """
    if not time_str:
        return None
    # Use regex to extract numbers for hours, minutes, and seconds
    m = re.match(r"(\d+)\s*h\s*(\d+)\s*m\s*(\d+)\s*s", time_str.strip(), re.IGNORECASE)
    if not m:
        return None
    h, mins, secs = int(m.group(1)), int(m.group(2)), int(m.group(3))
    total = h * 60 + mins
    # Apply rounding rule: seconds >= 30 rounds up
    if secs >= 30:
        total += 1
    return total

# Register the Python function as a PySpark UDF
parse_time_udf = F.udf(parse_resolution_time, IntegerType())

# Unit testing the parser on test inputs
print("Testing parser on sample cases:")
test_cases = [("0h 22m 45s", 23), ("0h 14m 20s", 14), ("1h 10m 30s", 71), ("0h 15m 00s", 15), ("BADTIME", None)]
for raw, expected in test_cases:
    res = parse_resolution_time(raw)
    status = "✅" if res == expected else "❌"
    print(f"   {status} parse({raw!r}) = {res} (Expected: {expected})")

🧪 Testing parser on sample cases:
   ✅ parse('0h 22m 45s') = 23 (Expected: 23)
   ✅ parse('0h 14m 20s') = 14 (Expected: 14)
   ✅ parse('1h 10m 30s') = 71 (Expected: 71)
   ✅ parse('0h 15m 00s') = 15 (Expected: 15)
   ✅ parse('BADTIME') = None (Expected: None)


/databricks/spark/python/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


### 3-C · Apply Time Conversion (Rule R-1, R-2)

In [0]:
def apply_time_conversion(df):
    return (
        df
        .withColumn("status_clean", F.upper(F.trim(F.col("status"))))
        .withColumn("resolved_minutes", parse_time_udf(F.col("resolution_time")))
        .filter(F.col("resolved_minutes").isNotNull())
    )

print("⏱️ Converting resolution times:")
silver_day1_times = apply_time_conversion(clean_day1)
silver_day2_times = apply_time_conversion(clean_day2)
print(f"   Day 1 rows after conversion: {silver_day1_times.count()}")
print(f"   Day 2 rows after conversion: {silver_day2_times.count()}")

⏱️ Converting resolution times:
   Day 1 rows after conversion: 1004
   Day 2 rows after conversion: 804


### 3-D · Join Profiles & Scope Filter (Rule R-4)
Keep only agents who report to Team Leads TL01 through TL08.

In [0]:
IN_SCOPE_TLS = {f"TL{str(i).zfill(2)}" for i in range(1, 9)}

def enrich_and_scope_filter(tickets_df, profiles_df):
    # Join with profiles to get agent metadata
    enriched = tickets_df.join(
        F.broadcast(profiles_df.select("agent_id", "agent_name", "role", "team_lead_id")),
        on="agent_id", 
        how="inner"
    )
    # Filter scope for Team Leads TL01 - TL08
    return enriched.filter(F.col("team_lead_id").isin(IN_SCOPE_TLS))

print("🔭 Filtering scope for TL01-TL08:")
silver_day1_scoped = enrich_and_scope_filter(silver_day1_times, bronze_profiles)
silver_day2_scoped = enrich_and_scope_filter(silver_day2_times, bronze_profiles)
print(f"   Day 1 in-scope rows: {silver_day1_scoped.count()}")
print(f"   Day 2 in-scope rows: {silver_day2_scoped.count()}")

🔭 Filtering scope for TL01-TL08:
   Day 1 in-scope rows: 953
   Day 2 in-scope rows: 761


### 3-E · Apply Quality Threshold (Rule R-3)
Ticket must be status `"RESOLVED"` and duration must be strictly greater than 15 minutes.

In [0]:
def apply_quality_threshold(df):
    return (
        df
        .withColumn("is_successful", (F.col("status_clean") == "RESOLVED") & (F.col("resolved_minutes") > 15))
        .filter(F.col("is_successful") == True)
    )

print("✅ Applying > 15 min Quality Threshold:")
silver_day1_success = apply_quality_threshold(silver_day1_scoped)
silver_day2_success = apply_quality_threshold(silver_day2_scoped)
print(f"   Day 1 qualifying tickets: {silver_day1_success.count()}")
print(f"   Day 2 qualifying tickets: {silver_day2_success.count()}")

✅ Applying > 15 min Quality Threshold:
   Day 1 qualifying tickets: 493
   Day 2 qualifying tickets: 332


### 3-F · Save Silver Tables to Azure Data Lake
We write our cleaned, intermediate Silver datasets back to the `silver` container in Delta format.

In [0]:
# Write Silver clean tables to Azure Storage
(silver_day1_success.write
 .format("delta")
 .mode("overwrite")
 .save(f"abfss://silver@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/day1_tickets_silver"))

(silver_day2_success.write
 .format("delta")
 .mode("overwrite")
 .save(f"abfss://silver@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/day2_tickets_silver"))

print("✅ Silver Delta tables written to ADLS successfully.")

✅ Silver Delta tables written to ADLS successfully.


In [0]:
display(
    spark.read.format("delta")
    .load(f"abfss://silver@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/day1_tickets_silver")
)

agent_id,ticket_id,status,resolution_time,category,Day,status_clean,resolved_minutes,agent_name,role,team_lead_id,is_successful
A042,TKT00002,Resolved,1h 17m 11s,Billing,1,RESOLVED,77,Agent_A042,Junior Support Agent,TL07,true
A039,TKT00005,Resolved,0h 49m 27s,Returns,1,RESOLVED,49,Agent_A039,Support Agent,TL07,true
A007,TKT00007,Resolved,2h 53m 16s,Returns,1,RESOLVED,173,Agent_A007,Senior Support Agent,TL02,true
A039,TKT00008,Resolved,2h 2m 11s,Returns,1,RESOLVED,122,Agent_A039,Support Agent,TL07,true
A014,TKT00011,Resolved,0h 53m 58s,Delivery,1,RESOLVED,54,Agent_A014,Senior Support Agent,TL03,true
A014,TKT00012,Resolved,1h 1m 1s,Delivery,1,RESOLVED,61,Agent_A014,Senior Support Agent,TL03,true
A005,TKT00014,Resolved,0h 18m 55s,General,1,RESOLVED,19,Agent_A005,Senior Support Agent,TL01,true
A026,TKT00016,Resolved,0h 35m 22s,Returns,1,RESOLVED,35,Agent_A026,Senior Support Agent,TL05,true
A048,TKT00018,Resolved,2h 5m 13s,Technical,1,RESOLVED,125,Agent_A048,Support Agent,TL08,true
A023,TKT00021,Resolved,2h 39m 24s,Technical,1,RESOLVED,159,Agent_A023,Senior Support Agent,TL04,true


In [0]:
print("Silver Day1:", silver_day1_success.count())
print("Silver Day2:", silver_day2_success.count())

Silver Day1: 493
Silver Day2: 332


## 🥇 Step 4 · Gold Layer (Business Logic & Aggregation)
In this final layer, we implement the carry-over rule to join our datasets, then calculate aggregates to answer the 4 business questions.

### 4-A · Day 2 Carry-over Rule (Rule R-6)
Agents who successfully resolved tickets on Day 1 should be completely excluded from Day 2 results to prevent double-counting.

In [0]:
# Get list of agents who successfully resolved tickets on Day 1
day1_successful_agents = silver_day1_success.select("agent_id").distinct()

# Exclude these agents from Day 2 using a left anti join
silver_day2_carryover = silver_day2_success.join(
    day1_successful_agents,
    on="agent_id",
    how="left_anti"
)

# Union Day 1 results and filtered Day 2 results
gold_combined = silver_day1_success.union(silver_day2_carryover)

print(f"🔁 Carry-over filter applied:")
print(f"   Day-1 successful agents excluded: {day1_successful_agents.count()}")
print(f"   Day-2 carry-over rows remaining : {silver_day2_carryover.count()}")
print(f"   Gold Combined total row count    : {gold_combined.count()}")

🔁 Carry-over filter applied:
   Day-1 successful agents excluded: 45
   Day-2 carry-over rows remaining : 12
   Gold Combined total row count    : 505


### 📊 Question 1 — Ticket Resolution Rates Across the Team Hierarchy
Sum of successfully resolved tickets and efficiency metrics grouped by each Team Lead.

In [0]:
gold_q1 = (
    gold_combined
    .groupBy("team_lead_id")
    .agg(
        F.count("ticket_id").alias("total_resolved_tickets"),
        F.countDistinct("agent_id").alias("active_agents"),
        F.round(F.count("ticket_id") / F.countDistinct("agent_id"), 2).alias("avg_tickets_per_agent")
    )
    .orderBy(F.col("total_resolved_tickets").desc())
)

print("Q1 Table Preview:")
gold_q1.show()

Q1 Table Preview:
+------------+----------------------+-------------+---------------------+
|team_lead_id|total_resolved_tickets|active_agents|avg_tickets_per_agent|
+------------+----------------------+-------------+---------------------+
|        TL05|                    74|            6|                12.33|
|        TL03|                    73|            6|                12.17|
|        TL06|                    72|            6|                 12.0|
|        TL07|                    68|            6|                11.33|
|        TL01|                    57|            6|                  9.5|
|        TL08|                    56|            6|                 9.33|
|        TL02|                    56|            6|                 9.33|
|        TL04|                    49|            6|                 8.17|
+------------+----------------------+-------------+---------------------+



### 📊 Question 2 — Per-Agent Performance for Day 1 and Day 2
Resolution counts for each agent across both days, showing their growth trend.

In [0]:
gold_q2_long = (
    gold_combined
    .groupBy("agent_id", "agent_name", "team_lead_id", "Day")
    .agg(F.count("ticket_id").alias("resolved_count"))
)

gold_q2_wide = (
    gold_q2_long
    .groupBy("agent_id", "agent_name", "team_lead_id")
    .pivot("Day", [1, 2])
    .agg(F.sum("resolved_count"))
    .withColumnRenamed("1", "day1_resolved")
    .withColumnRenamed("2", "day2_resolved")
    .fillna(0, subset=["day1_resolved", "day2_resolved"])
    .withColumn("total_resolved", F.col("day1_resolved") + F.col("day2_resolved"))
    .withColumn("trend",
        F.when(F.col("day2_resolved") == 0, "Day 1 Only")
         .when(F.col("day1_resolved") == 0, "Day 2 Carry-over")
         .when(F.col("day2_resolved") > F.col("day1_resolved"), "Improved ↑")
         .when(F.col("day2_resolved") < F.col("day1_resolved"), "Declined ↓")
         .otherwise("Stable →")
    )
    .orderBy("team_lead_id", F.col("total_resolved").desc())
)

print("Q2 Table Preview (Top 10 rows):")
gold_q2_wide.show(10)

Q2 Table Preview (Top 10 rows):
+--------+----------+------------+-------------+-------------+--------------+----------------+
|agent_id|agent_name|team_lead_id|day1_resolved|day2_resolved|total_resolved|           trend|
+--------+----------+------------+-------------+-------------+--------------+----------------+
|    A005|Agent_A005|        TL01|           13|            0|            13|      Day 1 Only|
|    A003|Agent_A003|        TL01|           12|            0|            12|      Day 1 Only|
|    A006|Agent_A006|        TL01|           10|            0|            10|      Day 1 Only|
|    A002|Agent_A002|        TL01|           10|            0|            10|      Day 1 Only|
|    A004|Agent_A004|        TL01|            0|            6|             6|Day 2 Carry-over|
|    A001|Agent_A001|        TL01|            6|            0|             6|      Day 1 Only|
|    A010|Agent_A010|        TL02|           12|            0|            12|      Day 1 Only|
|    A009|Agent_A0

### 📊 Question 3 — Compliance with the Resolution Quality Threshold
The percentage of tickets from each team that successfully passed the > 15-minute quality threshold.

In [0]:
# All tickets marked as "Resolved" (regardless of time taken)
all_resolved_inScope = (
    silver_day1_scoped.filter(F.col("status_clean") == "RESOLVED")
    .union(silver_day2_scoped.filter(F.col("status_clean") == "RESOLVED"))
)

total_resolved_all = (
    all_resolved_inScope
    .groupBy("team_lead_id")
    .agg(F.count("ticket_id").alias("total_resolved_any_time"))
)

# Total tickets that passed the quality threshold (> 15 min)
total_resolved_qual = (
    gold_combined
    .groupBy("team_lead_id")
    .agg(F.count("ticket_id").alias("qualifying_tickets"))
)

gold_q3 = (
    total_resolved_all
    .join(total_resolved_qual, on="team_lead_id", how="left")
    .fillna(0, subset=["qualifying_tickets"])
    .withColumn("compliance_rate_pct",
        F.round(F.col("qualifying_tickets") / F.col("total_resolved_any_time") * 100, 2))
    .orderBy(F.col("compliance_rate_pct").desc())
)

print("Q3 Table Preview:")
gold_q3.show()

Q3 Table Preview:
+------------+-----------------------+------------------+-------------------+
|team_lead_id|total_resolved_any_time|qualifying_tickets|compliance_rate_pct|
+------------+-----------------------+------------------+-------------------+
|        TL05|                    110|                74|              67.27|
|        TL03|                    119|                73|              61.34|
|        TL07|                    116|                68|              58.62|
|        TL06|                    127|                72|              56.69|
|        TL01|                    102|                57|              55.88|
|        TL02|                    109|                56|              51.38|
|        TL08|                    113|                56|              49.56|
|        TL04|                     99|                49|              49.49|
+------------+-----------------------+------------------+-------------------+



### 📊 Question 4 — Identifying Agents Who Carried Over Unresolved Work
The list of agents who struggled on Day 1 (no success) but resolved qualifying tickets on Day 2.

In [0]:
silver_day2_scoped.select(
    "agent_id",
    "status",
    "resolution_time",
    "resolved_minutes"
).show(20, False)

+--------+--------+---------------+----------------+
|agent_id|status  |resolution_time|resolved_minutes|
+--------+--------+---------------+----------------+
|A037    |Pending |1h 10m 46s     |71              |
|A005    |Pending |2h 29m 10s     |149             |
|A012    |Resolved|0h 8m 37s      |9               |
|A004    |Resolved|0h 26m 33s     |27              |
|A013    |Open    |1h 20m 8s      |80              |
|A032    |Resolved|2h 22m 15s     |142             |
|A027    |Open    |0h 6m 2s       |6               |
|A002    |Resolved|1h 45m 2s      |105             |
|A023    |Resolved|0h 27m 45s     |28              |
|A017    |Pending |1h 21m 36s     |82              |
|A033    |Open    |2h 41m 58s     |162             |
|A027    |Resolved|0h 16m 10s     |16              |
|A020    |Resolved|0h 7m 23s      |7               |
|A022    |Resolved|0h 52m 50s     |53              |
|A014    |Pending |2h 54m 18s     |174             |
|A029    |Open    |0h 43m 32s     |44         

In [0]:
gold_q4 = (
    silver_day2_carryover
    .groupBy("agent_id", "agent_name", "team_lead_id")
    .agg(
        F.count("ticket_id").alias("day2_qualifying_tickets"),
        F.round(F.avg("resolved_minutes"), 2).alias("avg_resolution_mins_day2"),
        F.min("resolved_minutes").alias("min_resolution_mins"),
        F.max("resolved_minutes").alias("max_resolution_mins")
    )
    .orderBy("team_lead_id", F.col("day2_qualifying_tickets").desc())
)

print("Q4 Table Preview (Top 10 rows):")
gold_q4.show(10)

Q4 Table Preview (Top 10 rows):
+--------+----------+------------+-----------------------+------------------------+-------------------+-------------------+
|agent_id|agent_name|team_lead_id|day2_qualifying_tickets|avg_resolution_mins_day2|min_resolution_mins|max_resolution_mins|
+--------+----------+------------+-----------------------+------------------------+-------------------+-------------------+
|    A004|Agent_A004|        TL01|                      6|                    59.0|                 18|                172|
|    A011|Agent_A011|        TL02|                      2|                    44.0|                 34|                 54|
|    A021|Agent_A021|        TL04|                      4|                  102.75|                 33|                160|
+--------+----------+------------+-----------------------+------------------------+-------------------+-------------------+



### 💾 Step 5 · Save Gold Aggregated Tables to ADLS Gen2
Save the final calculated Gold datasets into our `gold` container in Delta format so that Power BI can fetch them.

In [0]:
(gold_q1.write
 .format("delta")
 .mode("overwrite")
 .save(f"abfss://gold@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold_q1"))

(gold_q2_wide.write
 .format("delta")
 .mode("overwrite")
 .save(f"abfss://gold@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold_q2"))

(gold_q3.write
 .format("delta")
 .mode("overwrite")
 .save(f"abfss://gold@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold_q3"))

(gold_q4.write
 .format("delta")
 .mode("overwrite")
 .save(f"abfss://gold@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold_q4"))

print("🥇 All Gold tables successfully written to Azure Storage in Delta format!")

🥇 All Gold tables successfully written to Azure Storage in Delta format!


## 🏁 Execution Summary
We print a high-level summary of the pipeline's execution.

In [0]:
total_qualifying  = gold_combined.count()
unique_agents_q   = gold_combined.select("agent_id").distinct().count()
unique_tls_q      = gold_combined.select("team_lead_id").distinct().count()
carryover_count   = gold_q4.count()

print("=" * 55)
print("  PIPELINE EXECUTION SUMMARY")
print("=" * 55)
print(f"  Qualifying resolved tickets  : {total_qualifying}")
print(f"  Contributing agents          : {unique_agents_q}")
print(f"  Team Leads in scope          : {unique_tls_q} (TL01–TL08)")
print(f"  Day 2 carry-over agents      : {carryover_count}")
print("=" * 55)